# 05b · Weekly GTM Infographic
One-page visual summary for GTM leadership. Run all cells — the final cell renders the HTML report.

Adjust `week_override` in the setup cell to target a specific week.

In [ ]:
import datetime, base64, io, re
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display, HTML
from snowflake.snowpark.context import get_active_session
plt.switch_backend("agg")

SF_BLUE  = "#29B5E8"; SF_TEAL = "#00A9CE"; SF_GREEN = "#36B37E"
SF_ORANGE= "#FF8B00"; SF_DARK = "#0A2859"; SF_GRAY  = "#E8EDF2"; BG = "#F4F7FA"
ACT = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"
session = get_active_session()

week_override = None
today = datetime.date.today()
if week_override:
    week_start = week_override - datetime.timedelta(days=week_override.weekday())
else:
    week_start = today - datetime.timedelta(days=today.weekday() + 7)
week_end   = week_start + datetime.timedelta(days=6)
prev_start = week_start - datetime.timedelta(days=7)
prev_end   = week_start - datetime.timedelta(days=1)
ws, we, ps, pe = str(week_start), str(week_end), str(prev_start), str(prev_end)

TEAM_IDS = [
    "005VI00000Z0y6bYAB","005VI00000ExC3VYAV","0053r00000BblkZAAR",
    "005VI00000HajknYAB","0050Z000009IrKRQA0","005VI00000XibQfYAJ",
]
team_ids_str = "', '".join(TEAM_IDS)

def fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=130, facecolor=fig.get_facecolor())
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

print(f"Report week: {ws} to {we}")

In [ ]:
%%sql -r this_week
SELECT
    COUNT(*)                                                     AS meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID)                                AS accounts,
    SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))                     AS setsail_matched,
    ROUND(100.0*SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))/NULLIF(COUNT(*),0),0) AS setsail_pct,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))                    AS with_uc,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='No' AND SF_ACCOUNT_ID IS NOT NULL,1,0)) AS uc_gap,
    COUNT(DISTINCT MEETING_SE_NAME) AS active_specialists
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'

In [ ]:
%%sql -r prev_week
SELECT COUNT(*) AS meetings, COUNT(DISTINCT SF_ACCOUNT_ID) AS accounts
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES WHERE MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'

In [ ]:
%%sql -r new_accounts
SELECT CUSTOMER, SF_ACCOUNT_ID, MEETING_SE_NAME AS specialist,
       MIN(MEETING_DATE) AS first_touch_ever, MAX(MEETING_DATE) AS this_week_date
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE SF_ACCOUNT_ID IS NOT NULL AND CUSTOMER IS NOT NULL
GROUP BY 1,2,3
HAVING MIN(MEETING_DATE) BETWEEN '{{ws}}' AND '{{we}}'
ORDER BY this_week_date

In [ ]:
%%sql -r long_running
SELECT CUSTOMER, SF_ACCOUNT_ID,
       MIN(MEETING_DATE) AS first_touch, MAX(MEETING_DATE) AS last_touch,
       COUNT(*) AS total_meetings,
       COUNT(DISTINCT DATE_TRUNC('week',MEETING_DATE)) AS active_weeks,
       MAX(USE_CASE_NAME) AS use_case, MAX(USE_CASE_TAGGED_IN_SF) AS uc_status
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE CUSTOMER IS NOT NULL AND SF_ACCOUNT_ID IS NOT NULL
GROUP BY 1,2
HAVING active_weeks >= 4 AND MAX(MEETING_DATE) >= DATEADD('day', -21, '{{we}}')
ORDER BY total_meetings DESC LIMIT 10

In [ ]:
%%sql -r pipeline
SELECT d.USE_CASE_STAGE, d.STAGE_NUMBER,
       COUNT(DISTINCT d.USE_CASE_ID) AS use_cases,
       ROUND(SUM(d.USE_CASE_EACV)/1e6,2) AS eacv_m,
       SUM(IFF(d.DECISION_DATE <= DATEADD('day',30,CURRENT_DATE()),1,0)) AS due_30d
FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.USE_CASE_ID = d.USE_CASE_ID
WHERE a.USER_ID IN ('{{team_ids_str}}') AND a.TEAM_ROLE IS NOT NULL
  AND d.IS_LOST=FALSE AND d.STAGE_NUMBER BETWEEN 1 AND 6
GROUP BY 1,2 ORDER BY 2

In [ ]:
%%sql -r top_accounts
SELECT CUSTOMER, COUNT(*) AS mtgs,
       MAX(USE_CASE_NAME) AS use_case,
       MAX(USE_CASE_TAGGED_IN_SF) AS uc_status,
       DATEDIFF('day',MAX(MEETING_DATE),CURRENT_DATE()) AS days_since
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'
  AND CUSTOMER IS NOT NULL AND CUSTOMER != ''
GROUP BY 1 ORDER BY 2 DESC LIMIT 12

In [ ]:
%%sql -r week_summaries
SELECT CUSTOMER, MEETING_SE_NAME, SUMMARY, NEXT_STEPS
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'
  AND SUMMARY IS NOT NULL AND TRIM(SUMMARY) != ''

In [ ]:
%%sql -r unique_customers
SELECT
    CUSTOMER,
    LISTAGG(DISTINCT MEETING_SE_NAME, ', ') WITHIN GROUP (ORDER BY MEETING_SE_NAME) AS specialists,
    COUNT(*) AS meetings,
    MAX(USE_CASE_TAGGED_IN_SF) AS uc_status
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ws}}' AND '{{we}}'
  AND CUSTOMER IS NOT NULL AND CUSTOMER != ''
GROUP BY 1
ORDER BY CUSTOMER

In [ ]:
%%sql -r tech_wins
WITH team_attribution AS (
    SELECT a.USE_CASE_ID,
           LISTAGG(DISTINCT n.ACTIVITY_SE_NAME, ', ') WITHIN GROUP (ORDER BY n.ACTIVITY_SE_NAME) AS team_specialists
    FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
    JOIN SALES.SE_REPORTING.DIM_SE_ACTIVITY n ON a.USER_ID = n.ACTIVITY_SE_ID
    WHERE a.USER_ID IN ('{{team_ids_str}}')
      AND a.TEAM_ROLE IS NOT NULL
    GROUP BY 1
)
SELECT DISTINCT
    d.ACCOUNT_NAME,
    d.USE_CASE_NAME,
    CASE
        WHEN d.IS_TECH_WON AND d.DECISION_DATE BETWEEN '{{ws}}' AND '{{we}}' THEN 'Tech Win'
        WHEN d.IS_WON     AND d.DECISION_DATE BETWEEN '{{ws}}' AND '{{we}}' THEN 'UC Win'
        WHEN d.IS_DEPLOYED AND d.ACTUAL_USE_CASE_DEPLOYMENT_DATE BETWEEN '{{ws}}' AND '{{we}}' THEN 'Go Live'
    END AS milestone,
    COALESCE(d.DECISION_DATE, d.ACTUAL_USE_CASE_DEPLOYMENT_DATE) AS milestone_date,
    ROUND(d.USE_CASE_EACV/1e3, 0) AS eacv_k,
    t.team_specialists AS specialists
FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.USE_CASE_ID = d.USE_CASE_ID
JOIN team_attribution t ON a.USE_CASE_ID = t.USE_CASE_ID
WHERE a.USER_ID IN ('{{team_ids_str}}')
  AND a.TEAM_ROLE IS NOT NULL
  AND (
    (d.IS_TECH_WON AND d.DECISION_DATE BETWEEN '{{ws}}' AND '{{we}}')
    OR (d.IS_WON   AND d.DECISION_DATE BETWEEN '{{ws}}' AND '{{we}}')
    OR (d.IS_DEPLOYED AND d.ACTUAL_USE_CASE_DEPLOYMENT_DATE BETWEEN '{{ws}}' AND '{{we}}')
  )
ORDER BY milestone, milestone_date DESC

In [ ]:
for df_name in ["this_week","prev_week","new_accounts","long_running","pipeline","top_accounts","week_summaries","unique_customers","tech_wins"]:
    v = locals().get(df_name) or globals().get(df_name)
    if v is not None and hasattr(v,"to_pandas"):
        globals()[df_name] = v.to_pandas()

tw  = this_week.iloc[0]
pw  = prev_week.iloc[0]
mtgs    = int(tw.get("MEETINGS",0) or 0)
accts   = int(tw.get("ACCOUNTS",0) or 0)
setsail = int(tw.get("SETSAIL_PCT",0) or 0)
uc_gap  = int(tw.get("UC_GAP",0) or 0)
new_ct  = len(new_accounts)
delta_m = mtgs - int(pw.get("MEETINGS",0) or 0)
delta_a = accts - int(pw.get("ACCOUNTS",0) or 0)
total_eacv = float(pipeline["EACV_M"].sum()) if len(pipeline) else 0
total_ucs  = int(pipeline["USE_CASES"].sum()) if len(pipeline) else 0
due_30     = int(pipeline["DUE_30D"].sum()) if len(pipeline) else 0


clrs = [SF_BLUE,SF_TEAL,SF_GREEN,SF_ORANGE,'#9B59B6','#E67E22']

# Chart 2: pipeline
fig2, ax = plt.subplots(figsize=(5.5,3)); fig2.patch.set_facecolor(BG); ax.set_facecolor(BG)
for s in ax.spines.values(): s.set_visible(False)
if len(pipeline):
    labels = pipeline["USE_CASE_STAGE"].str.replace(r"\d+ - ","",regex=True).str[:15]
    bars = ax.bar(labels, pipeline["USE_CASES"], color=clrs[:len(pipeline)], width=0.55)
    ax2  = ax.twinx(); ax2.set_facecolor(BG)
    for s in ax2.spines.values(): s.set_visible(False)
    ax2.plot(labels, pipeline["EACV_M"], color=SF_DARK, marker="o", linewidth=2, markersize=5)
    ax2.set_ylabel("eACV ($M)", fontsize=8, color=SF_DARK); ax2.tick_params(labelsize=8, colors=SF_DARK)
    for bar, val in zip(bars, pipeline["USE_CASES"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, str(int(val)), ha="center", fontsize=9, color=SF_DARK, fontweight="bold")
ax.set_title("Pipeline by Stage", fontsize=11, color=SF_DARK, pad=8)
ax.tick_params(axis="x", labelsize=8, rotation=15, colors=SF_DARK); ax.tick_params(axis="y", labelsize=8, colors=SF_DARK)
ax.set_ylabel("Use Cases", fontsize=9, color=SF_DARK)
plt.tight_layout(pad=1.2); CHART_PIPE = fig_to_b64(fig2)
print("Charts built")

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
texts = "\n---\n".join(
    f"[{r['MEETING_SE_NAME']} @ {r['CUSTOMER']}] {r['SUMMARY']}"
    for _, r in week_summaries.iterrows() if r["SUMMARY"]
)
ai_themes = ""
if texts.strip():
    prompt = (
        f"Week of {ws}. Gong call summaries from a Snowflake specialist team. "
        "In 3 bullet points (max 15 words each), what are the top 3 customer themes? "
        "Start each bullet with a bold topic keyword. No preamble.\n\n" + texts[:4000]
    )
    ai_themes = session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', ?)", [prompt]).collect()[0][0]
    ai_themes = "\n".join(l for l in ai_themes.strip().splitlines() if l.strip())
print("AI themes done")

In [ ]:
def kpi(value, label, color, delta=None, sub=None):
    dh = ""
    if delta is not None:
        sign = "&#9650;" if delta >= 0 else "&#9660;"
        dc   = "#27AE60" if delta >= 0 else "#E74C3C"
        dh   = f"<div style='font-size:11px;color:{dc};margin-top:2px'>{sign} {abs(delta)} vs prev wk</div>"
    sh = f"<div style='font-size:10px;color:#7f8c8d;margin-top:3px'>{sub}</div>" if sub else ""
    return (
        f"<div style='background:white;border-radius:10px;padding:18px 20px;text-align:center;"
        f"box-shadow:0 2px 8px rgba(0,0,0,0.08);border-top:4px solid {color};min-width:120px;flex:1'>"
        f"<div style='font-size:36px;font-weight:800;color:{color};line-height:1'>{value}</div>"
        f"<div style='font-size:11px;color:#0A2859;font-weight:600;margin-top:6px;"
        f"text-transform:uppercase;letter-spacing:.5px'>{label}</div>"
        + dh + sh + "</div>"
    )

# Build table rows
acct_rows = ""
for _, r in top_accounts.iterrows():
    uc = str(r.get("UC_STATUS",""))
    bg = "#eafaf1" if uc=="Yes" else "#fef9e7" if uc=="No" else "white"
    uc_badge = ("<span style='background:" + ("#d5f5e3" if uc=="Yes" else "#fdebd0") + ";"
                "color:" + ("#1e8449" if uc=="Yes" else "#d35400") + ";"
                "padding:2px 8px;border-radius:10px;font-size:11px;font-weight:600'>" + uc + "</span>")
    acct_rows += (
        "<tr style='background:" + bg + "'>"
        "<td style='padding:8px 12px;font-size:12px;font-weight:600;color:#2c3e50'>" + str(r['CUSTOMER']) + "</td>"
        "<td style='padding:8px 12px;font-size:12px;color:#7f8c8d;text-align:center'>" + str(int(r['MTGS'])) + "</td>"
        "<td style='padding:8px 12px;font-size:11px;color:#555'>" + ((r['USE_CASE'] or '')[:35] or "&mdash;") + "</td>"
        "<td style='padding:8px 12px;text-align:center'>" + (uc_badge if uc else "&mdash;") + "</td></tr>"
    )

new_rows = "".join(
    "<tr style='background:" + ("#e8f4fd" if i%2==0 else "white") + "'>"
    "<td style='padding:7px 12px;font-size:12px;color:#2c3e50'>" + str(r['CUSTOMER']) + "</td>"
    "<td style='padding:7px 12px;font-size:12px;color:#7f8c8d'>" + str(r['SPECIALIST']) + "</td></tr>"
    for i,(_, r) in enumerate(new_accounts.iterrows())
) or "<tr><td colspan='2' style='padding:10px;color:#aaa;font-size:12px'>None this week</td></tr>"

long_rows = "".join(
    "<tr style='background:" + ("#fff8f0" if i%2==0 else "white") + "'>"
    "<td style='padding:7px 12px;font-size:12px;font-weight:600;color:#2c3e50'>" + str(r['CUSTOMER']) + "</td>"
    "<td style='padding:7px 12px;font-size:12px;color:#7f8c8d;text-align:center'>" + str(int(r['ACTIVE_WEEKS'])) + " wks</td>"
    "<td style='padding:7px 12px;font-size:12px;color:#7f8c8d;text-align:center'>" + str(int(r['TOTAL_MEETINGS'])) + "</td>"
    "<td style='padding:7px 12px;font-size:12px;color:#2c3e50'>" + ((r['USE_CASE'] or '')[:40] or "&mdash;") + "</td></tr>"
    for i,(_, r) in enumerate(long_running.iterrows())
) or "<tr><td colspan='4' style='padding:10px;color:#aaa;font-size:12px'>None this week</td></tr>"

# AI themes bullets (top 3)
themes_html = ""
if ai_themes:
    for line in ai_themes.splitlines():
        line = line.strip().lstrip("-*\u2022").strip()
        if line:
            line = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", line)
            themes_html += "<li style='margin-bottom:8px;font-size:13px;color:#2c3e50;line-height:1.5'>" + line + "</li>"
else:
    themes_html = "<li style='color:#aaa;font-size:12px'>No Gong summaries this week</li>"

# Unique customers tag cloud
unique_count = len(unique_customers)
customer_tags = ""
for _, r in unique_customers.iterrows():
    uc     = str(r.get("UC_STATUS",""))
    bg     = "#eafaf1" if uc=="Yes" else "#eaf4fb" if uc=="Unknown" else "#fef9e7"
    border = "#27ae60" if uc=="Yes" else "#29B5E8" if uc=="Unknown" else "#f39c12"
    customer_tags += (
        "<span style='display:inline-block;background:" + bg + ";border:1px solid " + border + ";"
        "border-radius:16px;padding:4px 12px;margin:4px;font-size:12px;color:#2c3e50'>"
        "<strong>" + str(r['CUSTOMER']) + "</strong>"
        "<span style='color:#7f8c8d;font-size:10px;margin-left:6px'>" + str(r['SPECIALISTS']) + "</span>"
        "</span>"
    )
unique_section_html = (
    "<div class='sec' style='margin-bottom:18px'>"
    + "<div class='ttl'>All Unique Customers This Week (" + str(unique_count) + ")</div>"
    + "<div style='line-height:2.2'>" + customer_tags + "</div>"
    + "<div style='font-size:10px;color:#aaa;margin-top:10px'>"
    "Green border = use case tagged &nbsp;|&nbsp; Yellow = no use case</div></div>"
)

# Tech wins
wins_html = ""
milestone_colors = {
    "Tech Win": ("1a73e8","e8f0fe","&#127942;"),
    "UC Win":   ("0a9396","e0f4f4","&#9989;"),
    "Go Live":  ("1e8449","eafaf1","&#128640;"),
}
if len(tech_wins) == 0:
    wins_html = "<p style='color:#aaa;font-size:13px;margin:8px 0'>No tech wins, UC wins, or go-lives this week.</p>"
else:
    for _, r in tech_wins.iterrows():
        m = str(r.get("MILESTONE",""))
        color, bg, icon = milestone_colors.get(m, ("555555","f8f9fa","&#11088;"))
        eacv = ("$" + str(int(r['EACV_K'])) + "K") if r.get("EACV_K") else ""
        se   = str(r.get("SPECIALISTS","") or "")
        wins_html += (
            "<div style='background:#" + bg + ";border-left:5px solid #" + color + ";"
            "border-radius:8px;padding:14px 18px;margin-bottom:12px;display:flex;align-items:center;gap:16px'>"
            "<div style='font-size:28px'>" + icon + "</div>"
            "<div style='flex:1'>"
            "<div style='font-size:14px;font-weight:700;color:#0A2859'>" + str(r['ACCOUNT_NAME']) + "</div>"
            "<div style='font-size:12px;color:#555;margin-top:2px'>" + str(r['USE_CASE_NAME']) + "</div>"
            "</div>"
            "<div style='text-align:right'>"
            "<span style='background:#" + color + ";color:white;padding:3px 10px;"
            "border-radius:12px;font-size:11px;font-weight:700'>" + m + "</span>"
            + ("<div style='font-size:12px;color:#27ae60;font-weight:700;margin-top:4px'>" + eacv + "</div>" if eacv else "")
            + ("<div style='font-size:10px;color:#7f8c8d;margin-top:2px'>&#128100; " + se + "</div>" if se else "")
            + "</div></div>"
        )
wins_section_html = (
    "<div class='sec' style='margin-bottom:18px'>"
    + "<div class='ttl'>&#127942; Tech Wins &amp; Go-Lives This Week (" + str(len(tech_wins)) + ")</div>"
    + wins_html + "</div>"
)

# Derived metrics
team_size   = int(tw.get("ACTIVE_SPECIALISTS", 0) or 6)
mtgs        = int(tw.get("MEETINGS",0) or 0)
accts       = int(tw.get("ACCOUNTS",0) or 0)
setsail     = int(tw.get("SETSAIL_PCT",0) or 0)
uc_gap      = int(tw.get("UC_GAP",0) or 0)
new_ct      = len(new_accounts)
delta_m     = mtgs - int(pw.get("MEETINGS",0) or 0)
delta_a     = accts - int(pw.get("ACCOUNTS",0) or 0)
total_eacv  = float(pipeline["EACV_M"].sum()) if len(pipeline) else 0
total_ucs   = int(pipeline["USE_CASES"].sum()) if len(pipeline) else 0
due_30      = int(pipeline["DUE_30D"].sum()) if len(pipeline) else 0

HTML_OUT = (
"<!DOCTYPE html><html><head><meta charset='utf-8'>"
"<style>"
"body{font-family:'Segoe UI',Arial,sans-serif;background:#F4F7FA;margin:0;padding:20px}"
".sec{background:white;border-radius:12px;padding:20px 24px;margin-bottom:18px;box-shadow:0 2px 8px rgba(0,0,0,0.07)}"
".ttl{font-size:12px;font-weight:700;color:#0A2859;text-transform:uppercase;letter-spacing:.8px;"
"margin-bottom:14px;border-bottom:2px solid #29B5E8;padding-bottom:6px;display:inline-block}"
"table{width:100%;border-collapse:collapse}"
"th{background:#0A2859;color:white;padding:9px 12px;font-size:11px;text-align:left;font-weight:600;letter-spacing:.4px}"
"</style></head><body>"

"<div style='background:linear-gradient(135deg,#0A2859 0%,#1a4a8a 100%);border-radius:14px;"
"padding:24px 32px;margin-bottom:20px;display:flex;justify-content:space-between;align-items:center'>"
"<div>"
"<div style='color:#29B5E8;font-size:11px;font-weight:700;letter-spacing:1px;text-transform:uppercase'>Field Enterprise Data Architecture Team</div>"
"<div style='color:white;font-size:26px;font-weight:800;margin:4px 0'>Weekly GTM Update</div>"
f"<div style='color:#a8c8e8;font-size:13px'>Week of {ws} &mdash; {we}</div>"
f"<div style='color:#29B5E8;font-size:12px;margin-top:6px'>&#128101; {team_size} active specialists</div>"
"</div>"
"<div style='text-align:right'>"
f"<div style='color:white;font-size:32px;font-weight:800'>${total_eacv:.1f}M</div>"
"<div style='color:#a8c8e8;font-size:11px;text-transform:uppercase;letter-spacing:.5px'>Active Pipeline eACV</div>"
f"<div style='color:#FF8B00;font-size:13px;font-weight:600;margin-top:4px'>{total_ucs} Use Cases &nbsp;|&nbsp; {due_30} due in 30d</div>"
"</div></div>"

"<div style='display:flex;gap:14px;margin-bottom:18px;flex-wrap:wrap'>"
+ kpi(mtgs,   "Meetings",     SF_BLUE,   delta=delta_m)
+ kpi(accts,  "Accounts",     SF_TEAL,   delta=delta_a)
+ kpi(new_ct, "New Accounts", SF_GREEN,  sub="first touch ever")
+ kpi(str(setsail)+"%", "Setsail Match", SF_ORANGE if setsail<60 else SF_TEAL, sub="calendar to SF")
+ kpi(uc_gap, "UC Gap",       "#E74C3C" if uc_gap>3 else SF_ORANGE, sub="need use case")
+ "</div>"

"<div class='sec' style='margin-bottom:18px'>"
"<div class='ttl'>&#128161; Top Customer Themes This Week</div>"
f"<ul style='margin:8px 0;padding-left:20px'>{themes_html}</ul>"
f"<div style='font-size:10px;color:#aaa;margin-top:10px'>Generated from {len(week_summaries)} Gong summaries &nbsp;&middot;&nbsp; Powered by Snowflake Cortex</div>"
"</div>"

+ wins_section_html

+ "<div style='display:flex;gap:16px;margin-bottom:18px'>"
"<div class='sec' style='flex:1'>"
"<div class='ttl'>Pipeline Health</div>"
f"<img src='data:image/png;base64,{CHART_PIPE}' style='width:100%;border-radius:6px'>"
"</div></div>"

+ "<div style='display:flex;gap:16px;margin-bottom:18px'>"
"<div class='sec' style='flex:2'>"
"<div class='ttl'>Top Accounts This Week</div>"
"<table><tr><th>Account</th><th>Mtgs</th><th>Use Case</th><th>UC Status</th></tr>"
+ acct_rows + "</table></div>"
+ "<div class='sec' style='flex:1'>"
f"<div class='ttl'>New Accounts ({new_ct})</div>"
"<table><tr><th>Account</th><th>Specialist</th></tr>"
+ new_rows + "</table></div></div>"

+ "<div class='sec' style='margin-bottom:18px'>"
"<div class='ttl'>Long-Running Engagements (4+ active weeks)</div>"
"<table><tr><th>Account</th><th>Active Weeks</th><th>Total Meetings</th><th>Use Case</th></tr>"
+ long_rows + "</table></div>"

+ unique_section_html

+ "</body></html>"
)

display(HTML(HTML_OUT))

import os
os.makedirs("output", exist_ok=True)
fname = f"output/weekly_gtm_report_{ws}.html"
with open(fname, "w") as f:
    f.write(HTML_OUT)
print(f"Saved to {fname}")
